# PY500 — Hypothesis Testing

Hypothesis testing is the **scientific method for data** — you state a claim, collect evidence, and decide whether the evidence supports or refutes the claim.

## Topics Covered
1. Null vs Alternative Hypothesis
2. p-value, Significance Level (α), and Power
3. **Z-Test** — large samples, known variance
4. **t-Test** — small samples, unknown variance (1-sample, 2-sample, paired)
5. **Chi-Square Test** — categorical data (independence, goodness of fit)
6. Type I and Type II errors

## Key Concepts
| Term | Meaning |
|---|---|
| **H₀ (Null)** | "No effect" — the default assumption |
| **H₁ (Alternative)** | "There IS an effect" — what you want to prove |
| **p-value** | Probability of seeing this data (or more extreme) IF H₀ is true |
| **α (Significance)** | Threshold for rejecting H₀ (typically 0.05) |
| **Power (1-β)** | Probability of correctly detecting a real effect |

> **Perspective:** A p-value is NOT the probability that H₀ is true. It is the probability of the *data* assuming H₀. This is the most common misinterpretation in statistics. Think of it as: "How surprising is this data if nothing is going on?"

---
## 1. Z-Test — Large Samples, Known Variance

In [ ]:
## One-sample Z-Test
# Question: Is the average salary in our sample different from ₹60,000?

import numpy as np
from scipy import stats

np.random.seed(42)
sample = np.random.normal(63000, 12000, 100)  # True mean = 63000

# H₀: μ = 60000 (population mean is 60K)
# H₁: μ ≠ 60000 (two-tailed test)

pop_mean = 60000
pop_std = 12000       # Assume known (Z-test requirement)
n = len(sample)

z_stat = (sample.mean() - pop_mean) / (pop_std / np.sqrt(n))
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))   # Two-tailed

alpha = 0.05

print(f"Sample mean  : ₹{sample.mean():,.0f}")
print(f"Z-statistic  : {z_stat:.3f}")
print(f"p-value      : {p_value:.4f}")
print(f"α            : {alpha}")
print(f"Decision     : {'Reject H₀' if p_value < alpha else 'Fail to reject H₀'}")
print(f"Interpretation: {'Evidence suggests mean ≠ ₹60K' if p_value < alpha else 'No evidence against mean = ₹60K'}")

# Learning Point: Z-Test requires knowing the population std (σ).
# In practice, you rarely know σ — that is why the t-Test exists.

---
## 2. t-Test — The Workhorse of Hypothesis Testing

In [ ]:
## One-sample t-Test
# Question: Is the average salary different from ₹60,000?
# (Same question but now we DON'T know population std)

t_stat, p_value = stats.ttest_1samp(sample, popmean=60000)

print("=== One-sample t-Test ===")
print(f"t-statistic  : {t_stat:.3f}")
print(f"p-value      : {p_value:.4f}")
print(f"Decision     : {'Reject H₀' if p_value < 0.05 else 'Fail to reject H₀'}")

In [ ]:
## Independent two-sample t-Test
# Question: Do IT and HR departments have different average salaries?

np.random.seed(42)
it_salaries = np.random.normal(75000, 15000, 50)
hr_salaries = np.random.normal(58000, 12000, 45)

# H₀: μ_IT = μ_HR (no difference)
# H₁: μ_IT ≠ μ_HR (two-tailed)

t_stat, p_value = stats.ttest_ind(it_salaries, hr_salaries)

print("=== Independent Two-Sample t-Test ===")
print(f"IT mean      : ₹{it_salaries.mean():,.0f} (n={len(it_salaries)})")
print(f"HR mean      : ₹{hr_salaries.mean():,.0f} (n={len(hr_salaries)})")
print(f"t-statistic  : {t_stat:.3f}")
print(f"p-value      : {p_value:.6f}")
print(f"Decision     : {'Reject H₀ — salaries differ' if p_value < 0.05 else 'Fail to reject H₀'}")

In [ ]:
## Paired t-Test
# Question: Did a training program improve employee ratings?
# Same employees measured BEFORE and AFTER.

np.random.seed(42)
before = np.random.normal(3.5, 0.5, 30)
after = before + np.random.normal(0.3, 0.4, 30)   # Small improvement + noise

t_stat, p_value = stats.ttest_rel(after, before)   # Paired test

print("=== Paired t-Test ===")
print(f"Before mean  : {before.mean():.2f}")
print(f"After mean   : {after.mean():.2f}")
print(f"Avg change   : {(after - before).mean():.2f}")
print(f"t-statistic  : {t_stat:.3f}")
print(f"p-value      : {p_value:.4f}")
print(f"Decision     : {'Training improved ratings' if p_value < 0.05 else 'No significant improvement'}")

# Learning Point: Use PAIRED t-test when the same subjects are measured
# twice (before/after). It controls for individual differences.

---
## 3. Chi-Square Test — Categorical Data

In [ ]:
## Chi-Square Test of Independence
# Question: Is department assignment independent of gender?

import pandas as pd

# Contingency table (observed frequencies)
observed = pd.DataFrame(
    [[30, 10, 5],
     [20, 25, 10]],
    index=['Male', 'Female'],
    columns=['IT', 'HR', 'Finance']
)

print("=== Observed Frequencies ===")
print(observed)

chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"\n=== Expected Frequencies (if independent) ===")
print(pd.DataFrame(expected.round(1), index=observed.index, columns=observed.columns))

print(f"\nChi² statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value       : {p_value:.4f}")
print(f"Decision      : {'Reject H₀ — dept and gender are related' if p_value < 0.05 else 'No evidence of relationship'}")

In [ ]:
## Chi-Square Goodness of Fit
# Question: Does our dice follow a fair distribution?

observed_counts = [18, 22, 16, 20, 14, 10]  # Actual rolls (100 total)
expected_counts = [100/6] * 6                 # Fair dice expectation

chi2, p_value = stats.chisquare(observed_counts, f_exp=expected_counts)

print("=== Goodness of Fit ===")
print(f"Observed: {observed_counts}")
print(f"Expected: {[round(e, 1) for e in expected_counts]}")
print(f"Chi²    : {chi2:.3f}")
print(f"p-value : {p_value:.4f}")
print(f"Decision: {'Dice is NOT fair' if p_value < 0.05 else 'No evidence dice is unfair'}")

---
## 4. Type I and Type II Errors, Power

| | H₀ is True | H₀ is False |
|---|---|---|
| **Reject H₀** | Type I Error (α) — False Positive | Correct! (Power = 1-β) |
| **Fail to reject H₀** | Correct! | Type II Error (β) — False Negative |

- **Type I (α):** Concluding an effect exists when it doesn't (like a fire alarm with no fire)
- **Type II (β):** Missing a real effect (like a fire alarm that doesn't go off)
- **Power:** Probability of detecting a real effect = 1 - β (aim for ≥ 0.8)

---
## Which Test When?

| Scenario | Test |
|---|---|
| One mean vs known value, large n, σ known | **Z-Test** |
| One mean vs known value, σ unknown | **One-sample t-Test** |
| Compare means of 2 independent groups | **Independent t-Test** |
| Compare means of same group before/after | **Paired t-Test** |
| Compare means of 3+ groups | **ANOVA** (see notebook 04) |
| Test independence of 2 categorical variables | **Chi-Square Independence** |
| Test if observed fits expected distribution | **Chi-Square Goodness of Fit** |
| Non-normal data, 2 groups | **Mann-Whitney U** (non-parametric) |

> **Best Practice:** Always check assumptions before running a test: normality (Shapiro-Wilk test), equal variances (Levene's test). If assumptions are violated, use non-parametric alternatives or bootstrap methods.